# UPDATE_BOXER (José)

Test notebook: it installs pandas, creates a temporary boxers database and runs `update_boxer`. Just press **Run All**.

In [25]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import pandas as pd
import sqlite3

#Make sure nothing is duplicated when you run the code again
connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("""
  CREATE TABLE boxers(
    id INTEGER PRIMARY KEY,
    name TEXT,
    total_fights INTEGER,
    wins INTEGER,
    loses INTEGER,
    draws INTEGER
  )
""")

dataframe = pd.DataFrame(
  [
    ("Mike Tyson", 59, 50, 7, 2),
    ("Roy Jones Jr.", 76, 66, 10, 0),
    ("Sugar Ray Leonard", 40, 36, 3, 1),
    ("Julio Cesar Chavez", 115, 107, 6, 2),
    ("Muhammad Ali", 61, 56, 5, 0),
    ("Roberto Duran", 119, 103, 16, 0),
    ("Evander Holyfield", 57, 44, 10, 2),
    ("Floyd Mayweather Jr.", 50, 50, 0, 0),
    ("Anthony Joshua", 34, 30, 4, 0),
    ("Oleksandr Usyk", 25, 25, 0, 0),
  ],
  columns=["name", "total_fights", "wins", "loses", "draws"],
)
dataframe.to_sql("boxers", con=connection, if_exists="append", index_label="id")
dataframe


,name,total_fights,wins,loses,draws
0,Mike Tyson,59,50,7,2
1,Roy Jones Jr.,76,66,10,0
2,Sugar Ray Leonard,40,36,3,1
3,Julio Cesar Chavez,115,107,6,2
4,Muhammad Ali,61,56,5,0
5,Roberto Duran,119,103,16,0
6,Evander Holyfield,57,44,10,2
7,Floyd Mayweather Jr.,50,50,0,0
8,Anthony Joshua,34,30,4,0
9,Oleksandr Usyk,25,25,0,0


In [27]:
def update_boxer(boxer: str, field: str, new_value: int) -> pd.DataFrame | None:
  """
  Update the wins, loses or draws of a boxer and show the updated table.

  Looks up a boxer by name and replaces one of their records with a new value
  (e.g. wins from 50 to 51). The total_fights of that boxer is adjusted by the
  same difference, so the data stays consistent.

  Args:
    boxer (str): Name of the boxer, written as in the table (e.g. "Mike Tyson").
    field (str): Record to change. Only "wins", "loses" or "draws" are valid.
    new_value (int): New value for that record. Must be a whole number >= 0.
  Returns:
    pd.DataFrame | None: Pandas object with all the boxers after the change,
      or None if nothing was updated.

  Example:
    update_boxer("Mike Tyson", "wins", 51)
    Prints: Boxer updated successfully (wins: 50 -> 51)
    Returns the table with Mike Tyson now having 51 wins and 60 total_fights.
  """
  try:
    #Check that the field is valid
    if field not in ("wins", "loses", "draws"):
      print("Error: field must be 'wins', 'loses' or 'draws'")
      return None

    #Check that the new value is >= 0. Bool is also an int, so we exclude it also
    if isinstance(new_value, bool) or not isinstance(new_value, int) or new_value < 0:
      print("Error: the new value must be a whole number greater than or equal to 0")
      return None

    #Keep the old value of the boxer to show it later
    cursor.execute(f"SELECT {field} FROM boxers WHERE name = ?", (boxer,))
    row = cursor.fetchone()
    if row is None: #If the boxer doesnt exist, stop here and change nothing
      print("Boxer not found")
      return None
    old_value = row[0]

    #If it exists, change the record and move the "total_fights" by the same difference
    cursor.execute(
      f"UPDATE boxers SET {field} = ?, total_fights = total_fights + ? WHERE name = ?",
      (new_value, new_value - old_value, boxer)
    )
    connection.commit()

    print(f"Boxer updated successfully ({field}: {old_value} -> {new_value})")
    return pd.read_sql("SELECT name, total_fights, wins, loses, draws FROM boxers", connection)
  except Exception as error:  #If anything fails while doing the change, undo it
    connection.rollback()
    print("Unexpected error -> ", error) #Show the error that made it wrong 
    return None #Return None as nothing was updated

#Proof of the function
update_boxer("Mike Tyson", "wins", 51)
#update_boxer("Jose Dominguez", "wins", 20) --> Salta "Boxer not found"
connection.close()


Boxer updated successfully (wins: 50 -> 51)
